# 📊 Data Visualization Model Lab — Complete Practical Solutions

**Student:** Arul Agnes
**Course:** M.E. CSE — Data Visualization
**Date:** 2025
**Dataset:** `cse_student_performance_1200.csv` — 1,200 rows × 20 columns (SYNTHETIC data)

> All 10 lab questions are solved below: simple, readable code + interpretation after every chart.

## 📁 Setup and Dataset Loading

In [ ]:
# Install any missing packages (quiet = no wall of text)
!pip install umap-learn --quiet

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
print("Libraries loaded. pandas", pd.__version__, "| seaborn", sns.__version__)

In [ ]:
# Load the dataset - try every plausible path (Kaggle input, local, downloaded)
csv_path = None
candidates = [
    "/kaggle/input/cse-student-performance-1200/cse_student_performance_1200.csv",
    "/kaggle/input/cse-student-performance/cse_student_performance_1200.csv",
    "/kaggle/input/data-v/cse_student_performance_1200.csv",
    "cse_student_performance_1200.csv",
    "../datasets/cse_student_performance_1200.csv",
    "datasets/cse_student_performance_1200.csv",
]
for path in candidates:
    if os.path.exists(path):
        csv_path = path
        break

# last resort: download the public dataset from Kaggle Hub
if csv_path is None:
    try:
        import kagglehub
        folder = kagglehub.dataset_download("arulmariaagnes/cse-student-performance-1200")
        csv_path = os.path.join(folder, "cse_student_performance_1200.csv")
    except Exception as e:
        print("kagglehub fallback failed:", e)

if csv_path is None:
    print("CSV not found! Use 'Add Input' on Kaggle and attach")
    print("dataset: arulmariaagnes/cse-student-performance-1200")
else:
    df = pd.read_csv(csv_path)
    print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns from:")
    print(" ", csv_path)
    print(df.head())

## Question 1: Bar Chart, Histogram and Box Plot

**Concepts:** comparing categories (bar), distribution shape (histogram), spread + outliers (box plot).

In [ ]:
# --- Q1: three basic charts ---
sec_mean = df.groupby("section")["final_exam_marks"].mean()

# (1) bar chart: average marks per section, with value labels
plt.figure(figsize=(8, 5))
bars = plt.bar(sec_mean.index, sec_mean.values, color="#2196F3", edgecolor="black")
for bar, v in zip(bars, sec_mean.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f"{v:.1f}", ha="center", fontweight="bold")
plt.title("Average Final Exam Marks by Section")
plt.xlabel("Section"); plt.ylabel("Average Marks"); plt.ylim(0, 100)
plt.tight_layout(); plt.savefig("q01_bar_chart.png", dpi=150, bbox_inches="tight")
plt.show()

# (2) histogram of all final marks with a mean line
plt.figure(figsize=(8, 5))
plt.hist(df["final_exam_marks"], bins=20, color="#4CAF50", edgecolor="black", alpha=0.85)
m = df["final_exam_marks"].mean()
plt.axvline(m, color="red", linestyle="--", linewidth=2, label=f"Mean = {m:.1f}")
plt.title("Distribution of Final Exam Marks")
plt.xlabel("Marks"); plt.ylabel("Number of Students"); plt.legend()
plt.tight_layout(); plt.savefig("q01_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

# (3) box plot: spread of marks inside each section
plt.figure(figsize=(8, 5))
sns.boxplot(x="section", y="final_exam_marks", data=df, order=list("ABCD"),
            hue="section", palette="Set2", legend=False)
plt.title("Final Exam Marks Distribution by Section")
plt.xlabel("Section"); plt.ylabel("Marks")
plt.tight_layout(); plt.savefig("q01_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

spread = df.groupby("section")["final_exam_marks"].std()
print(f"Section {sec_mean.idxmax()} has the highest average ({sec_mean.max():.1f})")
print(f"Marks range from {df['final_exam_marks'].min():.1f} to {df['final_exam_marks'].max():.1f}")
print(f"Section {spread.idxmax()} varies the most (std = {spread.max():.1f})")

**Interpretation:** bar charts compare category averages, the histogram shows the roughly
bell-shaped spread of marks around the mean, and the box plot shows every section has a similar
median with a handful of low outliers.

## Question 2: Time-Series Line Chart

**Concepts:** datetime conversion, grouping by date, trends over time.

In [ ]:
# --- Q2: average marks on each assessment date ---
df["assessment_date"] = pd.to_datetime(df["assessment_date"])
by_date = (df.groupby("assessment_date")["final_exam_marks"].mean()
             .reset_index().sort_values("assessment_date"))

plt.figure(figsize=(12, 5))
plt.plot(by_date["assessment_date"], by_date["final_exam_marks"],
         marker="o", color="#2196F3", linewidth=2)
plt.title("Average Student Performance Over Semester")
plt.xlabel("Assessment Date"); plt.ylabel("Average Final Exam Marks")
plt.xticks(rotation=45); plt.grid(alpha=0.4)
plt.tight_layout(); plt.savefig("q02_time_series.png", dpi=150, bbox_inches="tight")
plt.show()

first, last = by_date["final_exam_marks"].iloc[0], by_date["final_exam_marks"].iloc[-1]
peak = by_date.loc[by_date["final_exam_marks"].idxmax()]
valley = by_date.loc[by_date["final_exam_marks"].idxmin()]
trend = "INCREASING" if last > first + 1 else ("DECREASING" if last < first - 1 else "STABLE")
print(f"Performance trend: {trend} ({first:.1f} -> {last:.1f})")
print(f"Highest average {peak['assessment_date'].date()} = {peak['final_exam_marks']:.1f}")
print(f"Lowest average  {valley['assessment_date'].date()} = {valley['final_exam_marks']:.1f}")

**Interpretation:** the line wobbles because each date assesses a *different* group of
students — treat single-date jumps as sampling noise unless they persist over several weeks.

## Question 3: Misleading Chart and Its Fix

**Concepts:** truncated axes, colour accessibility, honest design.

In [ ]:
# --- Q3: deliberately BAD chart vs honest chart ---
dept_mean = df.groupby("department")["final_exam_marks"].mean()

# BAD: axis starts at 55 (exaggerates), rainbow colours, no labels
plt.figure(figsize=(8, 5))
plt.bar(dept_mean.index, dept_mean.values, color=["#ff0000", "#00cc00", "#0066ff"],
        edgecolor="black")
plt.ylim(55, 60); plt.yticks([]); plt.xticks([])
plt.tight_layout(); plt.savefig("q03_misleading.png", dpi=150, bbox_inches="tight")
plt.show()

# GOOD: axis from 0, colourblind-safe colour, title + labels + value labels
plt.figure(figsize=(8, 5))
bars = plt.bar(dept_mean.index, dept_mean.values, color="#2196F3", edgecolor="black")
for bar, v in zip(bars, dept_mean.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
             f"{v:.1f}", ha="center", fontweight="bold")
plt.ylim(0, 100)
plt.title("Average Final Marks by Department")
plt.xlabel("Department"); plt.ylabel("Average Marks")
plt.tight_layout(); plt.savefig("q03_fixed.png", dpi=150, bbox_inches="tight")
plt.show()

print("Issue 1: truncated y-axis exaggerated tiny differences  -> Fix: axis starts at 0")
print("Issue 2: red/green is unreadable for colour-blind viewers -> Fix: single blue")
print("Issue 3: no title or labels made it uninterpretable       -> Fix: added both")
print("Issue 4: fake 3D/decoration distorts perception           -> Fix: clean 2D bars")
print(f"Real spread between departments: only {dept_mean.max()-dept_mean.min():.1f} marks")

**Interpretation:** the same numbers look like a 10× difference when the axis is truncated.
**Rule:** bar charts must start at 0.

## Question 4: Compare Two Graphical Encodings

**Concepts:** position/length vs size encoding, Cleveland–McGill hierarchy.

In [ ]:
# --- Q4: same data, two encodings ---
sec_stats = df.groupby("section").agg(marks=("final_exam_marks", "mean"),
                                      hours=("study_hours_per_week", "mean")).reset_index()
sec_stats["num"] = range(1, 5)

# encoding 1: bars (position + length)
plt.figure(figsize=(8, 5))
plt.bar(sec_stats["section"], sec_stats["marks"], color="#2196F3", edgecolor="black")
plt.ylim(0, 100); plt.title("Encoding 1: Bar Chart (Position + Length)")
plt.xlabel("Section"); plt.ylabel("Average Marks")
plt.tight_layout(); plt.savefig("q04_encoding_1.png", dpi=150, bbox_inches="tight")
plt.show()

# encoding 2: bubbles (size)
plt.figure(figsize=(8, 5))
plt.scatter(sec_stats["num"], sec_stats["hours"], s=sec_stats["marks"] * 12,
            c="#FF5722", alpha=0.65, edgecolors="black")
for x, y, m in zip(sec_stats["num"], sec_stats["hours"], sec_stats["marks"]):
    plt.annotate(f"{m:.1f}", (x, y), ha="center", va="center", color="white",
                 fontweight="bold")
plt.xticks(sec_stats["num"], sec_stats["section"])
plt.title("Encoding 2: Bubble Chart (Size)")
plt.xlabel("Section"); plt.ylabel("Average Study Hours / Week")
plt.tight_layout(); plt.savefig("q04_encoding_2.png", dpi=150, bbox_inches="tight")
plt.show()

# redesign: two honest bar charts
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].bar(sec_stats["section"], sec_stats["marks"], color="#2196F3", edgecolor="black")
ax[0].set(title="Average Marks", xlabel="Section", ylabel="Marks", ylim=(0, 100))
ax[1].bar(sec_stats["section"], sec_stats["hours"], color="#4CAF50", edgecolor="black")
ax[1].set(title="Average Study Hours", xlabel="Section", ylabel="Hours", ylim=(0, 20))
plt.tight_layout(); plt.savefig("q04_redesigned.png", dpi=150, bbox_inches="tight")
plt.show()

print("Bar chart wins: eyes compare LENGTH on a common scale far better than AREA.")
print("Cleveland-McGill ranking: position > length > angle > area > colour.")

**Interpretation:** bubble areas double misleadingly (Stevens' power law) — a 58 vs 62
bubble pair is nearly impossible to tell apart, while two bars make the gap obvious.

## Question 5: Exploratory Data Analysis

**Concepts:** distributions, skewness, IQR outliers, KDE/violin plots.

In [ ]:
# --- Q5: four views of the key variables ---
cols = ["attendance_percentage", "study_hours_per_week", "lab_marks",
        "internal_marks", "final_exam_marks"]
short = ["Attendance", "Study Hours", "Lab Marks", "Internal", "Final Marks"]

# histograms
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for i, c in enumerate(cols):
    ax = axes.ravel()[i]
    ax.hist(df[c], bins=20, color="#42A5F5", edgecolor="black", alpha=0.85)
    ax.set_title(short[i])
axes.ravel()[5].set_visible(False)
fig.suptitle("Histograms of Key Variables")
plt.tight_layout(); plt.savefig("q05_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

# box plots
plt.figure(figsize=(10, 5)); df[cols].boxplot(rot=15)
plt.title("Box Plots of Key Variables"); plt.ylabel("Value")
plt.tight_layout(); plt.savefig("q05_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()

# density plots
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for i, c in enumerate(cols):
    sns.kdeplot(df[c], ax=axes[i], fill=True, color="#2196F3")
    axes[i].set_title(short[i])
plt.tight_layout(); plt.savefig("q05_density.png", dpi=150, bbox_inches="tight")
plt.show()

# violin plots
long = df[cols].melt(var_name="variable", value_name="value")
plt.figure(figsize=(12, 5))
sns.violinplot(x="variable", y="value", data=long, palette="Set2", inner="quartile")
plt.title("Violin Plots of Key Variables"); plt.xticks(rotation=15)
plt.tight_layout(); plt.savefig("q05_violin.png", dpi=150, bbox_inches="tight")
plt.show()

# statistics table
for c in cols:
    q1, q3 = df[c].quantile(.25), df[c].quantile(.75)
    iqr = q3 - q1
    n_out = ((df[c] < q1 - 1.5*iqr) | (df[c] > q3 + 1.5*iqr)).sum()
    print(f"{c:26s} mean={df[c].mean():6.2f} median={df[c].median():6.2f} "
          f"std={df[c].std():5.2f} skew={df[c].skew():+5.2f} IQR-outliers={n_out}")

**Interpretation:** `study_hours` is right-skewed (few hard workers), `lab_marks` is
left-skewed (most do fine), and `final_exam_marks` is nearly symmetric — the classic bell shape.

## Question 6: Scatter Plot, Pair Plot and Correlation Heatmap

**Concepts:** relationships, correlation matrix, correlation ≠ causation.

In [ ]:
# --- Q6: relationships between variables ---
plt.figure(figsize=(8, 5))
sns.scatterplot(x="study_hours_per_week", y="final_exam_marks", hue="pass_status",
                data=df, palette={"Pass": "#4CAF50", "Fail": "#FF5722"}, alpha=0.7)
plt.title("Study Hours vs Final Marks")
plt.xlabel("Study Hours per Week"); plt.ylabel("Final Exam Marks")
plt.tight_layout(); plt.savefig("q06_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

g = sns.pairplot(df, hue="pass_status",
                 palette={"Pass": "#4CAF50", "Fail": "#FF5722"},
                 vars=["study_hours_per_week", "attendance_percentage",
                       "lab_marks", "final_exam_marks"], corner=True)
g.savefig("q06_pairplot.png", dpi=150, bbox_inches="tight")
plt.show()

num = df.select_dtypes(include="number")
corr = num.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            linewidths=0.5)
plt.title("Correlation Heatmap")
plt.tight_layout(); plt.savefig("q06_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

pairs = corr["final_exam_marks"].drop("final_exam_marks").sort_values(ascending=False)
print(f"Strongest positive: {pairs.index[0]} (r={pairs.iloc[0]:.2f})")
print(f"Weakest:            {pairs.index[-1]} (r={pairs.iloc[-1]:.2f})")
print("NOTE: Correlation does NOT prove causation!")

**Interpretation:** study hours show the strongest positive link with marks; support
sessions are negative because *struggling* students attend them. **Correlation ≠ causation.**

## Question 7: Linear Regression

**Concepts:** fit line, slope/intercept, R², residuals.

In [ ]:
# --- Q7: regress marks on study hours ---
X = df[["study_hours_per_week"]]
y = df["final_exam_marks"]
model = LinearRegression().fit(X, y)
slope, intercept, r2 = model.coef_[0], model.intercept_, model.score(X, y)
y_pred = model.predict(X)

plt.figure(figsize=(8, 5))
sns.regplot(x="study_hours_per_week", y="final_exam_marks", data=df, ci=95,
            scatter_kws={"alpha": 0.4, "color": "#2196F3"},
            line_kws={"color": "#FF5722"})
plt.title("Linear Regression: Study Hours -> Final Marks")
plt.xlabel("Study Hours per Week"); plt.ylabel("Final Exam Marks")
plt.text(0.05, 0.95, f"Slope = {slope:.2f}\nIntercept = {intercept:.1f}\nR2 = {r2:.3f}",
         transform=plt.gca().transAxes, va="top",
         bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
plt.tight_layout(); plt.savefig("q07_regression.png", dpi=150, bbox_inches="tight")
plt.show()

residuals = y - y_pred
plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.5, color="#4CAF50")
plt.axhline(0, color="red", linestyle="--", linewidth=2)
plt.title("Residual Plot"); plt.xlabel("Predicted Marks")
plt.ylabel("Residual (Actual - Predicted)")
plt.tight_layout(); plt.savefig("q07_residuals.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Slope: {slope:.2f} marks per extra study hour")
print(f"Intercept: {intercept:.1f} (predicted marks at 0 hours)")
print(f"R-squared: {r2:.3f} -> study hours explain {r2*100:.1f}% of the variance")
print("Limitation: correlation is not causation; residuals scattered = good fit.")

**Interpretation:** every extra study hour is worth ~0.9 marks on average, but R² ≈ 0.32
means ~68% of the variation comes from other factors (attendance, prior ability, luck…).

## Question 8: Rolling Average and Uncertainty Band

**Concepts:** smoothing, standard deviation band, trend vs noise.

In [ ]:
# --- Q8: rolling average + uncertainty band ---
weekly = (df.groupby("assessment_date")["final_exam_marks"]
            .agg(mean="mean", std="std").reset_index().sort_values("assessment_date"))
weekly["roll"] = weekly["mean"].rolling(3).mean()
weekly["roll_std"] = weekly["std"].rolling(3).std()

plt.figure(figsize=(12, 5))
plt.plot(weekly["assessment_date"], weekly["mean"], marker="o", linewidth=1.2,
         alpha=0.6, label="Weekly Mean")
plt.plot(weekly["assessment_date"], weekly["roll"], linewidth=2.5,
         color="#FF5722", label="3-Week Rolling Average")
plt.fill_between(weekly["assessment_date"],
                 weekly["roll"] - weekly["roll_std"],
                 weekly["roll"] + weekly["roll_std"],
                 alpha=0.3, color="#FF5722", label="+/-1 Std Dev Band")
plt.title("Performance Trend with Rolling Average and Uncertainty Band")
plt.xlabel("Assessment Date"); plt.ylabel("Average Final Exam Marks")
plt.xticks(rotation=45); plt.legend()
plt.tight_layout(); plt.savefig("q08_rolling_avg.png", dpi=150, bbox_inches="tight")
plt.show()

w = weekly.loc[weekly["std"].idxmax()]
print("Rolling average smooths weekly fluctuations to reveal the trend.")
print(f"Widest spread: {w['assessment_date'].date()} (std = {w['std']:.1f})")
print("The band is +/-1 standard deviation (data spread), NOT a confidence interval.")

**Interpretation:** wide bands mean the cohort tested that day was mixed; the rolling
line is the trend you should quote when someone asks "are marks improving?" 

## Question 9: PCA

**Concepts:** standardization, explained variance, dimensionality reduction.

In [ ]:
# --- Q9: PCA from 11 features to 2 dimensions ---
features = ["attendance_percentage", "study_hours_per_week", "lab_marks",
            "assignment_marks", "internal_marks", "previous_semester_cgpa",
            "online_assessment_score", "library_visits_per_month",
            "coding_practice_hours_per_week", "support_sessions_attended",
            "final_exam_marks"]
X = df[features]
print("Why standardize? PCA is scale-sensitive: 0-100 would dominate 0-10.")
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
emb = pca.fit_transform(X_scaled)
print(f"PC1 explained variance: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"PC2 explained variance: {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"Cumulative:            {pca.explained_variance_ratio_.sum()*100:.1f}%")

order = ["O", "A+", "A", "B+", "B", "C", "F"]
plt.figure(figsize=(9, 6))
sns.scatterplot(x=emb[:, 0], y=emb[:, 1], hue=df["final_grade"], hue_order=order,
                palette="viridis", alpha=0.75, s=45)
plt.title("PCA: Students in 2D Space (colored by Grade)")
plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
plt.legend(title="Grade", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.savefig("q09_pca_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7, 4))
plt.bar(["PC1", "PC2"], pca.explained_variance_ratio_ * 100,
        color=["#2196F3", "#4CAF50"], edgecolor="black")
plt.title("Explained Variance per Component"); plt.ylabel("Variance Explained (%)")
plt.ylim(0, 100)
plt.tight_layout(); plt.savefig("q09_variance.png", dpi=150, bbox_inches="tight")
plt.show()

print("Students with grade O cluster on the high side of PC1.")

**Interpretation:** PC1 is an "overall performance" axis — high marks, internals and
study hours push one way. Two PCs keep most but **not all** of the information.

## Question 10: t-SNE and UMAP

**Concepts:** non-linear embeddings, clusters, embedding pitfalls.

In [ ]:
# --- Q10: t-SNE (and UMAP if installed) ---
try:
    from umap import UMAP
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False

X_scaled = StandardScaler().fit_transform(df[features])
order = ["O", "A+", "A", "B+", "B", "C", "F"]

emb = TSNE(n_components=2, random_state=42, perplexity=30, init="pca").fit_transform(X_scaled)
plt.figure(figsize=(9, 6))
sns.scatterplot(x=emb[:, 0], y=emb[:, 1], hue=df["final_grade"], hue_order=order,
                palette="viridis", alpha=0.75, s=45)
plt.title("t-SNE: Student Performance Clusters")
plt.xlabel("t-SNE Dimension 1"); plt.ylabel("t-SNE Dimension 2")
plt.legend(title="Grade", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.savefig("q10_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

if UMAP_AVAILABLE:
    emb2 = UMAP(n_components=2, random_state=42, n_neighbors=15).fit_transform(X_scaled)
    plt.figure(figsize=(9, 6))
    sns.scatterplot(x=emb2[:, 0], y=emb2[:, 1], hue=df["final_grade"], hue_order=order,
                    palette="viridis", alpha=0.75, s=45)
    plt.title("UMAP: Student Performance Clusters")
    plt.xlabel("UMAP Dimension 1"); plt.ylabel("UMAP Dimension 2")
    plt.legend(title="Grade", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout(); plt.savefig("q10_umap.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("UMAP not installed -> run:  !pip install umap-learn")

print("t-SNE groups similar students together - but distances between clusters")
print("and cluster SIZES are NOT meaningful. Visualization only, never for clustering!")

**Interpretation:** similar grades clump together in both maps; UMAP keeps more of the
global layout, t-SNE sharper local islands. **Never measure distances on either map.**

## 📋 Summary and Key Learnings

1. **Bar / box / histogram** — the three basic views: categories, spread, shape.
2. **Time series** — convert dates first, then group; beware small per-date samples.
3. **Honesty** — zero-based axes, accessible colours, always title + label.
4. **Encoding** — position beats area beats colour (Cleveland–McGill).
5. **EDA** — mean/median/std/skew + IQR outliers tell the distribution story.
6. **Correlation** — read the heatmap, then say it out loud: *correlation ≠ causation*.
7. **Regression** — slope is "marks per hour", R² is "% of variance explained".
8. **Uncertainty** — std band = data spread; CI = precision of the mean.
9. **PCA** — standardize → components → check explained variance before trusting 2D.
10. **t-SNE / UMAP** — pretty pictures, meaningless distances; visualization only.

**Remember:** this dataset is **SYNTHETIC** — generated for learning, not real research.

In [ ]:
# Save everything that was produced
import glob
pngs = sorted(glob.glob("*.png"))
print(f"{len(pngs)} figures saved to {os.getcwd()}/")
for p in pngs:
    print(" -", p)
print("\nAll 10 experiments completed successfully!")